In [1]:
import pandas as pd

df = pd.read_csv("../data/cleaned_dataset_for_EDA.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Feature Engineering

## 1. Binary Encoding

We have converted our simple two-category variables into a 0/1 format. This allows the model to interpret these features as "flags."

* **Gender:** Female = 1, Male = 0
* **Yes/No Variables:** Yes = 1, No = 0

---

## 2. Simplification of Service Categories

As identified in our EDA, several features contain a "No internet service" value. To reduce dimensionality and noise, we will consolidate these into "No," as the absence of the service is the primary signal.

**Features to simplify:**

* OnlineSecurity
* OnlineBackup
* DeviceProtection
* TechSupport
* StreamingTV
* StreamingMovies

---

## 3. One-Hot Encoding for Multi-Class Features

For variables with more than two distinct categories (like `Contract` or `PaymentMethod`), we cannot simply use 0/1/2, as the model might incorrectly assume an order of importance (e.g., that 2 is "greater" than 0). Instead, we use **One-Hot Encoding** to create separate columns for each category.

* **Contract:** Creates `Contract_Month-to-month`, `Contract_One year`, `Contract_Two year`.
* **InternetService:** Creates `InternetService_DSL`, `InternetService_Fiber optic`, `InternetService_No`.

---

## 4. Feature Selection: Dropping the Identifier

The `customerID` column served its purpose for data tracking, but because it is unique to every row, it provides no generalizable pattern for the model to learn. We will remove it to prevent overfitting.

---

## 5. Summary of Data Transformation

| Transformation Type | Features Affected | Reasoning |
| --- | --- | --- |
| **Binary Mapping** | gender, Partner, Churn, etc. | Direct numerical representation for binary logic. |
| **Category Consolidation** | TechSupport, OnlineBackup, etc. | Merging "No internet service" with "No" to simplify the model. |
| **One-Hot Encoding** | Contract, PaymentMethod, etc. | Expanding categories into binary columns to avoid ordinal bias. |
| **Column Dropping** | customerID | Removing non-predictive unique identifiers. |

---


In [2]:
binary_cols = ["gender", "Partner", "Dependents", "PhoneService",
               "PaperlessBilling"] # Removed "Churn" as it's handled separately

for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0, "Female": 1, "Male": 0})

In [3]:
df[binary_cols].head()

,gender,Partner,Dependents,PhoneService,PaperlessBilling
0,1,1,0,0,1
1,0,0,0,1,0
2,0,0,0,1,1
3,0,0,0,0,0
4,1,0,0,1,1


In [4]:
# Simplify “No internet service” categories
internet_dependent_cols = ["OnlineSecurity", "OnlineBackup", 
                           "DeviceProtection", "TechSupport", 
                           "StreamingTV", "StreamingMovies"]

for col in internet_dependent_cols:
    df[col] = df[col].replace({"No internet service": "No"})

In [5]:
# One-Hot Encode multi-class features
# drop_first=True avoids dummy variable trap (redundant columns).
# This ensures numeric representation without implied order.
multi_class_cols = ["Contract", "InternetService", "PaymentMethod", 
                    "MultipleLines"]  # MultipleLines has "No phone service"

df = pd.get_dummies(df, columns=multi_class_cols, drop_first=True)

In [6]:
# Drop identifier column
df = df.drop(columns=["customerID"])
# Removes non-predictive ID to prevent overfitting.

In [7]:
internet_dependent_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                           "TechSupport", "StreamingTV", "StreamingMovies"]

for col in internet_dependent_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

In [8]:
# Verify final dataset
# Shape: confirm row × column dimensions
# Check binary columns: all 0/1
# Check numeric columns: no missing or string issues
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 25 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7032 non-null   int64  
 1   SeniorCitizen                          7032 non-null   int64  
 2   Partner                                7032 non-null   int64  
 3   Dependents                             7032 non-null   int64  
 4   tenure                                 7032 non-null   int64  
 5   PhoneService                           7032 non-null   int64  
 6   OnlineSecurity                         7032 non-null   int64  
 7   OnlineBackup                           7032 non-null   int64  
 8   DeviceProtection                       7032 non-null   int64  
 9   TechSupport                            7032 non-null   int64  
 10  StreamingTV                            7032 non-null   int64  
 11  Stre

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,Churn,Contract_One year,Contract_Two year,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,MultipleLines_No phone service,MultipleLines_Yes
0,1,0,1,0,1,0,0,1,0,0,...,No,False,False,False,False,False,True,False,True,False
1,0,0,0,0,34,1,1,0,1,0,...,No,True,False,False,False,False,False,True,False,False
2,0,0,0,0,2,1,1,1,0,0,...,Yes,False,False,False,False,False,False,True,False,False
3,0,0,0,0,45,0,1,0,1,1,...,No,True,False,False,False,False,False,False,True,False
4,1,0,0,0,2,1,0,0,0,0,...,Yes,False,False,True,False,False,True,False,False,False


## Final Feature Engineering and Dataset Validation (summary)

With this step, the dataset has been fully transformed from raw text and mixed types into a clean, mathematical matrix ready for machine learning.

---

## 1. Feature Engineering Summary

### Category Simplification

We successfully consolidated the "No internet service" label into "No" for six service-dependent columns.

* **Reasoning:** Since "No internet service" is logically equivalent to not having the add-on (Security, Backup, etc.), this consolidation reduces unnecessary complexity while retaining the core signal.

### One-Hot Encoding and the Dummy Variable Trap

We applied `pd.get_dummies` with `drop_first=True` to multi-class variables like `Contract` and `PaymentMethod`.

* **Reasoning:** By dropping the first category of each variable, we prevent **Multicollinearity** (the dummy variable trap). For example, if `Contract_One year` and `Contract_Two year` are both 0, the model mathematically knows the customer must be on a `Month-to-month` contract.

### Feature Selection

The `customerID` has been removed.

* **Reasoning:** Every value was unique, which would have forced a model to memorize individual instances rather than learning generalizable churn patterns (Overfitting).

---

## 2. Final Dataset Audit

Our dataset is now significantly different from the raw Kaggle download:

| Metric | Initial State | Current State |
| --- | --- | --- |
| **Row Count** | 7,043 | 7,032 (11 dropped due to 0 tenure) |
| **Column Count** | 21 | 25 (Expanded via One-Hot Encoding) |
| **Data Types** | Mixed (Object, Float, Int) | Pure Numeric (Int, Float, Bool) |
| **Missing Values** | Hidden (blank spaces) | 0 (Fully Cleaned) |

---

## 3. Visual Verification of Transformations

The `df.head()` confirms the successful mapping:

* **Binary Mapping:** `gender`, `Partner`, and `Churn` are now integers (0 or 1).
* **Service Flags:** `TechSupport`, `OnlineSecurity`, etc., are now binary.
* **Contract and Payment:** Represented by Boolean flags (True/False), which Python models treat as 1/0 during training.

---


In [9]:
df.to_csv("../data/data_for_model_building.csv", index=False)
print("Dataset saved as 'data_for_model_building.csv'!")

Dataset saved as 'data_for_model_building.csv'!
